### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [2]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


d:\CEIA\ml_env\.venv\Scripts\python.exe: No module named pip


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [3]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [4]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [5]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [6]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [7]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [8]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [9]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [10]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [11]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [12]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [13]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [14]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [15]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [16]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [17]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [18]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 1911, 1825, 1828], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [19]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [20]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [21]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [22]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](20,)","[480.,584.,591.,...,564.,465.,377.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](20,)","[-3.16,-2.96,-2.95,...,-3. ,-3.19,-3.4 ]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](20, 101631)","[[0. ,0.94,0. ,...,0. ,0. ,0. ], [1.39,0.6 ,0. ,...,0. ,0. ,0. ], [0.95,0.14,0. ,...,0. ,0. ,0. ], ..., [0.42,2.9 ,0.04,...,0. ,0. ,0. ], [0.61,1.36,0. ,...,0. ,0. ,0. ], [0.03,0.38,0. ,...,0. ,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](20, 101631)","[[-11.56,-10.9 ,-11.56,...,-11.56,-11.56,-11.56], [-10.69,-11.1 ,-11.56,...,-11.56,-11.56,-11.56], [-10.9 ,-11.44,-11.57,...,-11.57,-11.57,-11.57], ..., [-11.22,-10.21,-11.54,...,-11.57,-11.57,-11.57], [-11.09,-10.7 ,-11.56,...,-11.56,-11.56,-11.56], [-11.53,-11.23,-11.56,...,-11.56,-11.56,-11.56]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,101631


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [23]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [24]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


In [25]:
# TÚ CODIGO AQUÍ

## 1. Similaridad entre documentos

Tomamos 5 documentos al azar del conjunto de entrenamiento y medimos 
su similitud coseno contra todos los demás documentos vectorizados con TF-IDF.

La similitud coseno mide el ángulo entre dos vectores: vale 1 si apuntan 
en la misma dirección (muy similares) y 0 si son ortogonales (sin palabras 
en común). Es una métrica adecuada para textos porque ignora la longitud 
del documento y se enfoca en la distribución de términos.

Para cada documento analizamos:
- El contenido del texto
- Su etiqueta real
- Las etiquetas de sus 5 vecinos más cercanos

In [ ]:
# Fijamos semilla para reproducibilidad
np.random.seed(42)
doc_indices = np.random.choice(len(newsgroups_train.data), size=5, replace=False)

for idx in doc_indices:
    # Similitud coseno del documento contra todos los de train
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    
    # Los 5 más similares (excluyendo el propio documento, posición 0)
    top5_idx = np.argsort(cossim)[::-1][1:6]
    top5_sim = np.sort(cossim)[::-1][1:6]
    
    # Clase del documento original
    clase_original = newsgroups_train.target_names[y_train[idx]]
    
    print("=" * 70)
    print(f"Documento índice: {idx}")
    print(f"Clase: {clase_original}")
    print(f"\nTexto (primeras 150 caracteres):\n{newsgroups_train.data[idx][:300]}")
    print(f"\n--- 5 documentos más similares ---")
    
    for rank, (sim_idx, sim_val) in enumerate(zip(top5_idx, top5_sim), 1):
        clase_vecino = newsgroups_train.target_names[y_train[sim_idx]]
        print(f"\n  #{rank} | Similitud: {sim_val:.4f} | Clase: {clase_vecino}")
        print(f"  Texto: {newsgroups_train.data[sim_idx][:200]}")
    
    print()

Documento índice: 7492
Clase: comp.sys.mac.hardware

Texto (primeras 300 caracteres):
Could someone please post any info on these systems.

Thanks.
BoB
-- 
---------------------------------------------------------------------- 
Robert Novitskey | "Pursuing women is similar to banging one's head
rrn@po.cwru.edu  |  against a wall...with less opportunity for reward" 

--- 5 documentos más similares ---

  #1 | Similitud: 0.6665 | Clase: comp.sys.mac.hardware
  Texto: Hey everybody:

   I want to buy a mac and I want to get a good price...who doesn't?  So,
could anyone out there who has found a really good deal on a Centris 650
send me the price.  I don't want to k

  #2 | Similitud: 0.3476 | Clase: comp.sys.ibm.pc.hardware
  Texto: Hay all:

    Has anyone out there heard of any performance stats on the fabled p24t.
 I was wondering what it's performance compared to the 486/66 and/or
pentium would be.  Any info would be helpful.

  #3 | Similitud: 0.1799 | Clase: comp.sys.mac.hardware
  

### Interpretación de resultados

**Documento 7492 (comp.sys.mac.hardware):** El vecino más cercano 
pertenece a la misma clase con una similitud alta (0.666), lo cual 
tiene sentido ya que ambos textos tratan sobre hardware Mac. Sin embargo, 
el segundo vecino pertenece a comp.sys.ibm.pc.hardware, categoría 
temáticamente cercana que comparte vocabulario técnico de hardware 
(specs, performance, etc.). Los vecinos 3 y 5 vuelven a ser de la 
clase correcta. El caso #4 (misc.forsale) se explica por ser también 
una consulta corta sobre hardware. En general, la similitud funciona bien.

**Documento 3546 (comp.os.ms-windows.misc):** Todos los vecinos 
pertenecen a comp.sys.ibm.pc.hardware en lugar de la clase correcta. 
Esto se debe a que el documento habla de DMA (Direct Memory Access), 
término técnico muy presente en posts de hardware de PC. Es un caso 
donde el vocabulario compartido lleva a similitudes entre categorías 
relacionadas. TF-IDF no logra distinguir el contexto de uso del término.

**Documento 5582 (misc.forsale):** Los vecinos #1, #2 y #4 son de 
la misma clase correcta. El documento lista componentes de hardware 
en venta, y sus vecinos son posts similares de venta o búsqueda de 
hardware. Los vecinos #3 y #5 pertenecen a comp.graphics, 
probablemente por compartir términos técnicos como "monochrome monitor". 
La similitud funciona razonablemente bien.

**Documento 4793 (talk.politics.guns):** Los resultados son más 
dispersos: solo 2 de 5 vecinos comparten la clase correcta. 
El documento trata sobre regulación de armas en Canadá, y aparecen 
vecinos de sci.crypt y talk.politics.misc, posiblemente porque el 
vocabulario político y de debate es compartido entre estas categorías. 
Los valores de similitud son bajos (≈0.23), lo que indica que el 
documento es poco típico dentro de su clase.

**Documento 3813 (rec.sport.hockey):** Es el caso más problemático: 
ninguno de los 5 vecinos pertenece a la clase correcta, apareciendo 
alt.atheism, soc.religion.christian y sci.crypt. El texto es muy 
corto y descriptivo (habla de una máscara de portero), con poco 
vocabulario específico de hockey. Los valores de similitud bajos 
(≈0.23-0.25) sugieren que probablemente los vecinos son documentos 
largos y genéricos que dominan por volumen de palabras comunes.

**Conclusión general:** TF-IDF captura bien la similitud temática 
cuando los documentos tienen vocabulario específico y suficiente 
longitud. Los principales casos de error se dan en: (1) categorías 
técnicamente relacionadas con léxico solapado (ej: hardware de Mac 
vs PC), (2) documentos muy cortos con poco vocabulario discriminativo, 
y (3) categorías de debate general con vocabulario poco específico.

## 2. Clasificador por prototipos (Zero-Shot)

En lugar de entrenar un modelo, clasificamos cada documento de test 
buscando cuál documento de train le es más similar por similitud coseno, 
y asignándole esa etiqueta. Este enfoque se conoce como clasificación 
por el vecino más cercano (1-NN) y sirve como baseline interpretable.

A diferencia de Naïve Bayes, no aprende distribuciones de probabilidad: 
simplemente "recuerda" todos los ejemplos de entrenamiento y clasifica 
por similitud directa.

In [27]:
# Calculamos la similitud coseno entre TODOS los documentos de test 
# contra TODOS los de train de una sola vez
# shape resultante: (n_test, n_train)
sim_matrix = cosine_similarity(X_test, X_train)

# Para cada documento de test, obtenemos el índice del documento 
# de train más similar
best_match_idx = np.argmax(sim_matrix, axis=1)

# La predicción es la etiqueta del documento de train más similar
y_pred_proto = y_train[best_match_idx]

# Evaluamos con F1-score macro para comparar con el baseline de NB
score = f1_score(y_test, y_pred_proto, average='macro')
print(f"F1-Score Macro (clasificador por prototipos): {score:.4f}")
print(f"F1-Score Macro (MultinomialNB baseline):      0.5854")

F1-Score Macro (clasificador por prototipos): 0.5050
F1-Score Macro (MultinomialNB baseline):      0.5854


## 2. Interpretación - Clasificador por prototipos (Zero-Shot)

El clasificador por prototipos obtuvo un F1-Score Macro de **0.5050**, 
por debajo del baseline de Naïve Bayes (**0.5854**).

Esto es esperable por varias razones:

- **El clasificador 1-NN es sensible al ruido**: una sola coincidencia 
  léxica fuerte puede llevar a una predicción incorrecta, ya que no 
  considera la distribución global de la clase sino un único ejemplo.

- **Naïve Bayes aprende distribuciones**: al entrenarse sobre todos 
  los documentos de cada clase, captura patrones estadísticos más 
  robustos que la simple comparación con un vecino.

- **El espacio TF-IDF es de alta dimensionalidad**: en espacios muy 
  dispersos, la similitud coseno entre un documento de test y cualquier 
  documento individual de train puede ser poco representativa de la 
  clase en su conjunto.

Sin embargo, el resultado de 0.5050 es notable considerando que es un 
enfoque **zero-shot**: no requiere entrenamiento, es completamente 
interpretable (siempre podemos ver qué documento de train generó la 
predicción) y aun así alcanza un rendimiento razonablemente cercano 
al modelo entrenado.

La principal desventaja práctica es el **costo computacional**: requiere 
calcular la similitud contra todos los documentos de train para cada 
documento de test, lo cual escala mal con datasets grandes.

## 3. Optimización de modelos Naïve Bayes

El objetivo es superar el F1-Score Macro baseline de 0.5854 
experimentando con distintos parámetros del vectorizador y 
dos variantes del modelo: MultinomialNB y ComplementNB.

### Parámetros del TfidfVectorizer a explorar

- **min_df**: ignora términos que aparecen en menos de N documentos. 
  Reduce ruido y dimensionalidad.
- **max_df**: ignora términos que aparecen en más del X% de documentos. 
  Elimina palabras demasiado frecuentes y poco discriminativas.
- **sublinear_tf**: aplica 1 + log(tf) en lugar de tf. Reduce el peso 
  de términos muy frecuentes dentro de un documento.
- **stop_words**: elimina palabras vacías en inglés (the, is, are...) 
  que no aportan información temática.

### Sobre los modelos
- **MultinomialNB**: asume que las features siguen una distribución 
  multinomial. Adecuado para conteos y frecuencias de términos.
- **ComplementNB**: variante diseñada para datasets desbalanceados. 
  En lugar de modelar la probabilidad de cada clase, modela el 
  complemento, lo que suele dar mejores resultados en clasificación 
  de texto.

In [29]:
from itertools import product

# Definimos los parámetros a explorar
vectorizer_params = [
    {"min_df": 1, "max_df": 1.0, "sublinear_tf": False, "stop_words": None},
    {"min_df": 2, "max_df": 0.95, "sublinear_tf": True,  "stop_words": "english"},
    {"min_df": 3, "max_df": 0.90, "sublinear_tf": True,  "stop_words": "english"},
    {"min_df": 5, "max_df": 0.85, "sublinear_tf": True,  "stop_words": "english"},
    {"min_df": 2, "max_df": 1.0,  "sublinear_tf": True,  "stop_words": None},
]

modelos = {
    "MultinomialNB": MultinomialNB(),
    "ComplementNB":  ComplementNB(),
}

resultados = []

for vp in vectorizer_params:
    vec = TfidfVectorizer(**vp)
    X_tr = vec.fit_transform(newsgroups_train.data)
    X_te = vec.transform(newsgroups_test.data)
    
    for nombre_modelo, modelo in modelos.items():
        modelo.fit(X_tr, y_train)
        y_pred_exp = modelo.predict(X_te)
        score = f1_score(y_test, y_pred_exp, average='macro')
        
        resultados.append({
            "modelo": nombre_modelo,
            "min_df": vp["min_df"],
            "max_df": vp["max_df"],
            "sublinear_tf": vp["sublinear_tf"],
            "stop_words": vp["stop_words"],
            "f1_macro": score
        })
        
        print(f"[{nombre_modelo}] min_df={vp['min_df']} | max_df={vp['max_df']} | "
              f"sublinear_tf={vp['sublinear_tf']} | stop_words={vp['stop_words']} "
              f"→ F1: {score:.4f}")

# Mejor resultado
mejor = max(resultados, key=lambda x: x['f1_macro'])
print("\n" + "="*60)
print(f"Mejor configuración:")
print(f"  Modelo:       {mejor['modelo']}")
print(f"  min_df:       {mejor['min_df']}")
print(f"  max_df:       {mejor['max_df']}")
print(f"  sublinear_tf: {mejor['sublinear_tf']}")
print(f"  stop_words:   {mejor['stop_words']}")
print(f"  F1 Macro:     {mejor['f1_macro']:.4f}")
print(f"  Mejora sobre baseline: {mejor['f1_macro'] - 0.5854:+.4f}")

[MultinomialNB] min_df=1 | max_df=1.0 | sublinear_tf=False | stop_words=None → F1: 0.5854
[ComplementNB] min_df=1 | max_df=1.0 | sublinear_tf=False | stop_words=None → F1: 0.6930
[MultinomialNB] min_df=2 | max_df=0.95 | sublinear_tf=True | stop_words=english → F1: 0.6437
[ComplementNB] min_df=2 | max_df=0.95 | sublinear_tf=True | stop_words=english → F1: 0.6921
[MultinomialNB] min_df=3 | max_df=0.9 | sublinear_tf=True | stop_words=english → F1: 0.6465
[ComplementNB] min_df=3 | max_df=0.9 | sublinear_tf=True | stop_words=english → F1: 0.6907
[MultinomialNB] min_df=5 | max_df=0.85 | sublinear_tf=True | stop_words=english → F1: 0.6461
[ComplementNB] min_df=5 | max_df=0.85 | sublinear_tf=True | stop_words=english → F1: 0.6820
[MultinomialNB] min_df=2 | max_df=1.0 | sublinear_tf=True | stop_words=None → F1: 0.6016
[ComplementNB] min_df=2 | max_df=1.0 | sublinear_tf=True | stop_words=None → F1: 0.6932

Mejor configuración:
  Modelo:       ComplementNB
  min_df:       2
  max_df:       1.0
  

### Interpretación de resultados

Los experimentos muestran patrones claros y consistentes:

**ComplementNB supera sistemáticamente a MultinomialNB** en todas las 
configuraciones probadas. La diferencia es significativa: en la 
configuración base (sin ningún cambio de parámetros), ComplementNB 
obtiene 0.6930 vs 0.5854 de MultinomialNB. Esto se explica porque 
ComplementNB fue diseñado específicamente para clasificación de texto: 
al modelar el complemento de cada clase captura mejor las diferencias 
entre categorías, siendo más robusto ante el desbalance de clases 
presente en 20 Newsgroups.

**El parámetro más impactante fue el modelo en sí**, no los parámetros 
del vectorizador. Esto sugiere que la representación TF-IDF base ya 
es bastante buena, y que la ganancia principal viene de elegir el 
clasificador adecuado.

**sublinear_tf ayuda a MultinomialNB** (sube de 0.5854 a ~0.64) pero 
tiene efecto neutro o levemente negativo en ComplementNB. Esto tiene 
sentido: MultinomialNB es más sensible a términos con frecuencias muy 
altas, y la transformación logarítmica suaviza ese efecto.

**Filtrar vocabulario (min_df, max_df, stop_words) no mejora 
ComplementNB** y en algunos casos lo perjudica levemente. La 
configuración ganadora usa min_df=2 (elimina hápax, es decir términos 
que aparecen una sola vez) pero max_df=1.0 y sin stop_words, 
conservando el vocabulario más amplio posible. Filtrar demasiado 
(min_df=5, max_df=0.85) reduce el F1 de ComplementNB a 0.6820, 
indicando que se pierde información discriminativa.

**Resumen de mejoras sobre el baseline (0.5854):**

| Modelo        | Mejor F1 | Mejora   |
|---------------|----------|----------|
| MultinomialNB | 0.6465   | +0.0611  |
| ComplementNB  | 0.6932   | +0.1078  |

La mejor configuración global es **ComplementNB con min_df=2, 
max_df=1.0, sublinear_tf=True y sin stop_words**, logrando una 
mejora de +0.1078 sobre el baseline.

## 4. Similaridad entre palabras

Hasta ahora usamos la matriz documento-término (shape: n_documentos × n_palabras)
para representar documentos como vectores. Si la transponemos, obtenemos 
una matriz término-documento (shape: n_palabras × n_documentos) donde 
ahora cada fila es una palabra representada como un vector sobre todos 
los documentos en los que aparece.

Esta representación captura el contexto distribucional de cada palabra: 
dos palabras serán similares si aparecen en documentos similares. 
Es una forma simple de obtener embeddings de palabras sin necesidad 
de modelos neuronales.

Elegimos 5 palabras manualmente de distintas temáticas presentes 
en el dataset para obtener resultados interpretables:
- **car**: temática de autos (rec.autos)
- **god**: temática religiosa (alt.atheism, soc.religion.christian)
- **space**: temática espacial (sci.space)
- **gun**: temática política (talk.politics.guns)
- **windows**: temática informática (comp.os.ms-windows.misc)

In [30]:
# Transponemos la matriz documento-término
# shape: (n_palabras, n_documentos)
X_train_T = X_train.T

palabras = ["car", "god", "space", "gun", "windows"]

for palabra in palabras:
    # Verificamos que la palabra esté en el vocabulario
    if palabra not in tfidfvect.vocabulary_:
        print(f"'{palabra}' no está en el vocabulario\n")
        continue
    
    # Obtenemos el índice de la palabra en el vocabulario
    word_idx = tfidfvect.vocabulary_[palabra]
    
    # Calculamos similitud coseno de esa palabra contra todas las demás
    word_vec = X_train_T[word_idx]
    sim_words = cosine_similarity(word_vec, X_train_T)[0]
    
    # Top 5 más similares (excluyendo la palabra misma)
    top5_idx = np.argsort(sim_words)[::-1][1:6]
    top5_sim = np.sort(sim_words)[::-1][1:6]
    
    print("=" * 50)
    print(f"Palabra: '{palabra}'")
    print(f"{'Rank':<6} {'Palabra':<20} {'Similitud'}")
    print("-" * 50)
    for rank, (idx, sim) in enumerate(zip(top5_idx, top5_sim), 1):
        print(f"  #{rank}   {idx2word[idx]:<20} {sim:.4f}")
    print()

Palabra: 'car'
Rank   Palabra              Similitud
--------------------------------------------------
  #1   cars                 0.1797
  #2   criterium            0.1770
  #3   civic                0.1748
  #4   owner                0.1689
  #5   dealer               0.1681

Palabra: 'god'
Rank   Palabra              Similitud
--------------------------------------------------
  #1   jesus                0.2688
  #2   bible                0.2616
  #3   that                 0.2560
  #4   existence            0.2548
  #5   christ               0.2511

Palabra: 'space'
Rank   Palabra              Similitud
--------------------------------------------------
  #1   nasa                 0.3304
  #2   seds                 0.2966
  #3   shuttle              0.2928
  #4   enfant               0.2803
  #5   seti                 0.2465

Palabra: 'gun'
Rank   Palabra              Similitud
--------------------------------------------------
  #1   guns                 0.3582
  #2   crime       

### Interpretación de resultados

La transposición de la matriz documento-término produce representaciones 
de palabras basadas en su contexto distribucional: dos palabras son 
similares si co-aparecen en los mismos documentos. Los resultados 
muestran que este enfoque simple captura relaciones semánticas 
coherentes en la mayoría de los casos.

**'car' → cars, criterium, civic, owner, dealer**
Los vecinos son todos términos del mundo automotriz: modelos de autos 
(civic), roles en la compra/venta (owner, dealer) y términos de 
competición (criterium). Las similitudes son bajas (~0.17) porque 
"car" es una palabra muy frecuente que aparece en muchos documentos 
distintos, diluyendo su vector. Aun así, el campo semántico recuperado 
es correcto y coherente con la categoría rec.autos.

**'god' → jesus, bible, that, christ, existence**
Tres de los cinco vecinos son claramente del campo religioso/teológico 
(jesus, bible, christ), y "existence" aparece en debates filosóficos 
sobre la existencia de dios, típicos de alt.atheism y 
soc.religion.christian. La aparición de "that" como tercer vecino 
es un caso de ruido: es una palabra funcional muy frecuente que no 
debería aportar información semántica pero cuya alta frecuencia en 
documentos religiosos le genera similitud artificial. Sería filtrada 
con stop_words.

**'space' → nasa, seds, shuttle, enfant, seti**
Es el resultado más limpio y coherente: nasa, shuttle, seti y seds 
(Students for the Exploration and Development of Space) son todos 
términos específicos del dominio espacial con similitudes altas 
(~0.25-0.33). La aparición de "enfant" es llamativa y probablemente 
se debe a un documento particular que mezcla términos espaciales con 
texto en francés, un caso de ruido puntual. La categoría sci.space 
tiene vocabulario muy específico, lo que produce vectores de palabras 
más discriminativos y similitudes más altas.

**'gun' → guns, crime, handgun, homicides, firearms**
Resultado muy coherente semánticamente: todos los vecinos pertenecen 
al campo de armas y seguridad pública, reflejando el vocabulario 
típico de talk.politics.guns. Las similitudes son relativamente altas 
(~0.23-0.36), lo que indica que estos términos co-aparecen 
consistentemente en los mismos documentos. La presencia de "crime" 
y "homicides" captura además el contexto del debate político sobre 
control de armas, más allá de la mera sinonimia.

**'windows' → dos, ms, microsoft, nt, for**
Los vecinos dos, ms, microsoft y nt son todos términos del ecosistema 
Microsoft, coherentes con la categoría comp.os.ms-windows.misc. 
La aparición de "for" es otro caso de ruido por palabra funcional 
frecuente, similar a "that" en el caso de "god". Sería eliminada 
con stop_words. La alta similitud con "dos" refleja que en la época 
del dataset (1993) Windows y DOS eran frecuentemente discutidos juntos.

**Conclusión general**
La representación término-documento captura relaciones semánticas 
reales sin necesidad de modelos neuronales, simplemente por 
co-ocurrencia en documentos. Sus principales limitaciones son:

- **Sensible a palabras funcionales** (that, for) que generan 
  similitudes espurias y deberían filtrarse con stop_words.
- **Similitudes bajas para palabras frecuentes** (car, god) cuyos 
  vectores se diluyen al aparecer en muchos documentos distintos.
- **No captura relaciones sintácticas ni orden**: "gun control" y 
  "control gun" serían indistinguibles.
- A diferencia de word2vec o GloVe, no generaliza más allá de los 
  documentos vistos en entrenamiento.